# Traffic Light Management System

This notebook provides an interactive interface to run and compare different traffic light control algorithms using the MoST (Monaco SUMO Traffic) scenario.

In [1]:
# Import necessary libraries
import os
import sys
import time
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set plot style
plt.style.use('ggplot')

# Ensure visuals directory exists
os.makedirs('visuals', exist_ok=True)

In [2]:
# Path to SUMO configuration file with TraCI support
SUMO_CFG = 'MoSTScenario-master/scenario/most.traci.sumocfg'

# Available controllers
CONTROLLERS = {
    'Rule-Based': 'rule_based',
    'Adaptive': 'adaptive',
    'Q-Learning': 'q_learning',
    'DQN': 'dqn',
    'Multi-Agent': 'multi_agent'
}

# Learning-based controllers that can be trained
LEARNING_CONTROLLERS = {
    'Q-Learning': 'q_learning',
    'DQN': 'dqn',
    'Multi-Agent': 'multi_agent'
}

# Default simulation parameters from the config file
DEFAULT_START_TIME = 14400  # 4 hours in seconds
DEFAULT_END_TIME = 50400    # 14 hours in seconds
DEFAULT_STEP_LENGTH = 0.25   # 0.25 seconds to match config file

In [3]:
def train_model(controller_name, start_time, end_time, num_episodes, gui, progress_callback=None):
    """Train a model with the specified controller and parameters."""
    # Dynamically import the controller module
    controller_module = importlib.import_module(f'controllers.{controller_name}')
    
    # Check if the controller has a train function
    if not hasattr(controller_module, 'train'):
        raise NotImplementedError(f"The {controller_name} controller does not support training.")
    
    # Train the model
    metrics = controller_module.train(SUMO_CFG, start_time, end_time, num_episodes, gui=gui, progress_callback=progress_callback)
    
    return metrics

def run_simulation(controller_name, start_time, end_time, step_length, gui, training_mode=False):
    """Run a simulation with the specified controller and parameters."""
    # Dynamically import the controller module
    controller_module = importlib.import_module(f'controllers.{controller_name}')
    
    # Run the simulation
    metrics = controller_module.run(SUMO_CFG, start_time, end_time, gui=gui, training_mode=training_mode)
    
    return metrics

def plot_metrics(metrics, controller_name, save=False):
    """Plot the simulation metrics."""
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'Traffic Metrics - {controller_name}', fontsize=16)
    
    # Plot waiting times
    axs[0, 0].plot(metrics['waiting_times'])
    axs[0, 0].set_title('Waiting Times')
    axs[0, 0].set_xlabel('Simulation Step')
    axs[0, 0].set_ylabel('Total Waiting Time (s)')
    
    # Plot queue lengths
    axs[0, 1].plot(metrics['queue_lengths'])
    axs[0, 1].set_title('Queue Lengths')
    axs[0, 1].set_xlabel('Simulation Step')
    axs[0, 1].set_ylabel('Total Queue Length')
    
    # Plot emissions
    axs[1, 0].plot(metrics['emissions'])
    axs[1, 0].set_title('CO₂ Emissions')
    axs[1, 0].set_xlabel('Simulation Step')
    axs[1, 0].set_ylabel('Total CO₂ (mg/s)')
    
    # Plot average speeds
    axs[1, 1].plot(metrics['avg_speeds'])
    axs[1, 1].set_title('Average Speeds')
    axs[1, 1].set_xlabel('Simulation Step')
    axs[1, 1].set_ylabel('Average Speed (m/s)')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        # Save the figure
        filename = os.path.join('visuals', f'{controller_name}_metrics.png')
        plt.savefig(filename, dpi=300)
    
    return fig

def plot_training_metrics(metrics, controller_name, save=False):
    """Plot the training metrics."""
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Training Metrics - {controller_name}', fontsize=16)
    
    # Plot episode rewards
    if 'episode_rewards' in metrics:
        axs[0].plot(metrics['episode_rewards'])
        axs[0].set_title('Episode Rewards')
        axs[0].set_xlabel('Episode')
        axs[0].set_ylabel('Total Reward')
    
    # Plot episode waiting times
    if 'episode_waiting_times' in metrics:
        axs[1].plot(metrics['episode_waiting_times'])
        axs[1].set_title('Episode Average Waiting Times')
        axs[1].set_xlabel('Episode')
        axs[1].set_ylabel('Average Waiting Time (s)')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        # Save the figure
        filename = os.path.join('visuals', f'{controller_name}_training_metrics.png')
        plt.savefig(filename, dpi=300)
    
    return fig

def plot_comparison(all_metrics, save=False):
    """Plot a comparison of metrics from different controllers."""
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Controller Comparison', fontsize=16)
    
    # Calculate average metrics for each controller
    avg_metrics = {}
    for controller, metrics in all_metrics.items():
        avg_metrics[controller] = {
            'waiting_times': np.mean(metrics['waiting_times']),
            'queue_lengths': np.mean(metrics['queue_lengths']),
            'emissions': np.mean(metrics['emissions']),
            'avg_speeds': np.mean(metrics['avg_speeds'])
        }
    
    # Convert to DataFrame for easier plotting
    df = pd.DataFrame(avg_metrics).T
    
    # Plot waiting times
    df['waiting_times'].plot(kind='bar', ax=axs[0, 0])
    axs[0, 0].set_title('Average Waiting Times')
    axs[0, 0].set_ylabel('Average Waiting Time (s)')
    
    # Plot queue lengths
    df['queue_lengths'].plot(kind='bar', ax=axs[0, 1])
    axs[0, 1].set_title('Average Queue Lengths')
    axs[0, 1].set_ylabel('Average Queue Length')
    
    # Plot emissions
    df['emissions'].plot(kind='bar', ax=axs[1, 0])
    axs[1, 0].set_title('Average CO₂ Emissions')
    axs[1, 0].set_ylabel('Average CO₂ (mg/s)')
    
    # Plot average speeds
    df['avg_speeds'].plot(kind='bar', ax=axs[1, 1])
    axs[1, 1].set_title('Average Speeds')
    axs[1, 1].set_ylabel('Average Speed (m/s)')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save:
        # Save the figure
        filename = os.path.join('visuals', 'controller_comparison.png')
        plt.savefig(filename, dpi=300)
    
    return fig

In [4]:
# Create widgets for simulation parameters
mode_dropdown = widgets.Dropdown(
    options=['Train Models', 'Run with Pre-trained Models'],
    value='Run with Pre-trained Models',
    description='Mode:',
    style={'description_width': 'initial'}
)

controller_dropdown = widgets.Dropdown(
    options=list(CONTROLLERS.keys()),
    value=list(CONTROLLERS.keys())[0],
    description='Controller:',
    style={'description_width': 'initial'}
)

training_controller_dropdown = widgets.Dropdown(
    options=list(LEARNING_CONTROLLERS.keys()),
    value=list(LEARNING_CONTROLLERS.keys())[0],
    description='Controller:',
    style={'description_width': 'initial'}
)

start_time_slider = widgets.IntSlider(
    value=DEFAULT_START_TIME,
    min=0,
    max=86400,  # 24 hours in seconds
    step=3600,  # 1 hour in seconds
    description='Start Time (s):',
    style={'description_width': 'initial'}
)

end_time_slider = widgets.IntSlider(
    value=DEFAULT_START_TIME + 1200,  # Default to 20 minutes after start
    min=0,
    max=86400,  # 24 hours in seconds
    step=600,   # 10 minutes in seconds
    description='End Time (s):',
    style={'description_width': 'initial'}
)

step_length_slider = widgets.FloatSlider(
    value=DEFAULT_STEP_LENGTH,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Step Length (s):',
    style={'description_width': 'initial'}
)

num_episodes_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=50,
    step=1,
    description='Training Episodes:',
    style={'description_width': 'initial'}
)

gui_checkbox = widgets.Checkbox(
    value=False,
    description='Show GUI',
    style={'description_width': 'initial'}
)

# Create buttons
run_button = widgets.Button(
    description='Run Simulation',
    button_style='success',
    tooltip='Click to run the simulation'
)

train_button = widgets.Button(
    description='Train Model',
    button_style='warning',
    tooltip='Click to train the model'
)

compare_button = widgets.Button(
    description='Compare All Controllers',
    button_style='info',
    tooltip='Click to run all controllers and compare results'
)

# Create output widgets
output = widgets.Output()
progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

In [5]:
# Define button click handlers
all_metrics = {}

@output.capture()
def run_simulation_handler(b):
    """Handle the Run Simulation button click."""
    # Clear previous output
    clear_output(wait=True)
    output.clear_output()
    
    # Get selected parameters
    controller_name = CONTROLLERS[controller_dropdown.value]
    start_time = start_time_slider.value
    end_time = end_time_slider.value
    step_length = step_length_slider.value
    gui = gui_checkbox.value
    
    # Validate parameters
    if end_time <= start_time:
        print("Error: End time must be greater than start time.")
        return
    
    # Initialize progress bar
    progress_bar.value = 0
    progress_bar.max = end_time - start_time
    display(progress_bar)
    
    print(f"Running simulation with {controller_dropdown.value} controller...")
    print(f"Start time: {start_time} s, End time: {end_time} s, Step length: {step_length} s")
    print(f"GUI: {'Enabled' if gui else 'Disabled'}")
    print(f"Using pre-trained model (if available)")
    
    try:
        # Run the simulation
        metrics = run_simulation(controller_name, start_time, end_time, step_length, gui, training_mode=False)
        
        # Store metrics for comparison
        all_metrics[controller_dropdown.value] = metrics
        
        # Plot and save results
        fig = plot_metrics(metrics, controller_dropdown.value, save=True)
        display(fig)
        
        print(f"\nSimulation completed successfully!")
        print(f"Results saved to visuals/{controller_name}_metrics.png")
        
    except Exception as e:
        print(f"\nError running simulation: {str(e)}")
        import traceback
        traceback.print_exc()

In [6]:
@output.capture()
def train_model_handler(b):
    """Handle the Train Model button click."""
    # Clear previous output
    clear_output(wait=True)
    output.clear_output()
    
    # Get selected parameters
    controller_name = LEARNING_CONTROLLERS[training_controller_dropdown.value]
    start_time = start_time_slider.value
    end_time = end_time_slider.value
    num_episodes = num_episodes_slider.value
    gui = gui_checkbox.value
    
    # Validate parameters
    if end_time <= start_time:
        print("Error: End time must be greater than start time.")
        return
    
    # Initialize progress bar
    progress_bar.value = 0
    progress_bar.max = num_episodes
    display(progress_bar)
    
    print(f"Training {training_controller_dropdown.value} controller...")
    print(f"Start time: {start_time} s, End time: {end_time} s")
    print(f"Number of episodes: {num_episodes}")
    print(f"GUI: {'Enabled' if gui else 'Disabled'}")
    
    try:
        # Define progress callback function to update progress bar
        def update_progress(episode):
            progress_bar.value = episode
        
        # Train the model with progress callback
        metrics = train_model(controller_name, start_time, end_time, num_episodes, gui, progress_callback=update_progress)
        
        # Plot and save training results
        if metrics:
            fig = plot_training_metrics(metrics, controller_name, save=True)
            display(fig)
        
        print(f"\nTraining completed successfully!")
        print(f"Trained models saved to models/{controller_name}/")
        
    except Exception as e:
        print(f"\nError training model: {str(e)}")
        import traceback
        traceback.print_exc()

In [7]:
@output.capture()
def compare_all_controllers_handler(b):
    """Handle the Compare All Controllers button click."""
    # Clear previous output
    clear_output(wait=True)
    output.clear_output()
    
    # Get selected parameters
    start_time = start_time_slider.value
    end_time = end_time_slider.value
    step_length = step_length_slider.value
    gui = False  # Force GUI off for comparison
    
    # Validate parameters
    if end_time <= start_time:
        print("Error: End time must be greater than start time.")
        return
    
    print("Running comparison of all controllers...")
    print(f"Start time: {start_time} s, End time: {end_time} s, Step length: {step_length} s")
    print("Using pre-trained models for learning-based controllers (if available)")
    
    # Clear previous metrics
    all_metrics.clear()
    
    # Run each controller
    for controller_label, controller_name in CONTROLLERS.items():
        print(f"\nRunning {controller_label} controller...")
        
        # Initialize progress bar
        progress_bar.value = 0
        progress_bar.max = end_time - start_time
        display(progress_bar)
        
        try:
            # Run the simulation
            metrics = run_simulation(controller_name, start_time, end_time, step_length, gui, training_mode=False)
            
            # Store metrics for comparison
            all_metrics[controller_label] = metrics
            
            # Plot and save individual results
            fig = plot_metrics(metrics, controller_label, save=True)
            plt.close(fig)
            
            print(f"Completed {controller_label} controller simulation.")
            
        except Exception as e:
            print(f"Error running {controller_label} controller: {str(e)}")
            import traceback
            traceback.print_exc()
    
    # Plot comparison if we have metrics for at least two controllers
    if len(all_metrics) >= 2:
        print("\nGenerating comparison plot...")
        fig = plot_comparison(all_metrics, save=True)
        display(fig)
        print("Comparison plot saved to visuals/controller_comparison.png")
    else:
        print("\nNot enough controllers completed successfully for comparison.")

In [8]:
# Create the layout with boxes for each row
mode_box = widgets.HBox([mode_dropdown])
controller_box = widgets.HBox([controller_dropdown])
training_controller_box = widgets.HBox([training_controller_dropdown])
time_box = widgets.HBox([start_time_slider, end_time_slider])
step_box = widgets.HBox([step_length_slider, gui_checkbox])
num_episodes_box = widgets.HBox([num_episodes_slider])
run_button_box = widgets.HBox([run_button, compare_button])
train_button_box = widgets.HBox([train_button])
compare_button_box = widgets.HBox([compare_button])

In [9]:
# Function to update UI based on selected mode
def update_ui_for_mode(change):
    if change['new'] == 'Train Models':
        # Show training controls
        controller_box.layout.display = 'none'
        training_controller_box.layout.display = 'flex'
        num_episodes_box.layout.display = 'flex'
        run_button_box.layout.display = 'none'
        train_button_box.layout.display = 'flex'
        compare_button_box.layout.display = 'none'
    else:
        # Show simulation controls
        controller_box.layout.display = 'flex'
        training_controller_box.layout.display = 'none'
        num_episodes_box.layout.display = 'none'
        run_button_box.layout.display = 'flex'
        train_button_box.layout.display = 'none'
        compare_button_box.layout.display = 'flex'

# Attach handlers to buttons and dropdowns
run_button.on_click(run_simulation_handler)
train_button.on_click(train_model_handler)
compare_button.on_click(compare_all_controllers_handler)
mode_dropdown.observe(update_ui_for_mode, names='value')

In [11]:
# Create the main control panel
controls = widgets.VBox([
    mode_box,
    controller_box,
    training_controller_box,
    time_box,
    step_box,
    num_episodes_box,
    run_button_box,
    train_button_box
])

# Initialize UI based on default mode
if mode_dropdown.value == 'Train Models':
    controller_box.layout.display = 'none'
    training_controller_box.layout.display = 'flex'
    num_episodes_box.layout.display = 'flex'
    run_button_box.layout.display = 'none'
    train_button_box.layout.display = 'flex'
else:
    controller_box.layout.display = 'flex'
    training_controller_box.layout.display = 'none'
    num_episodes_box.layout.display = 'none'
    run_button_box.layout.display = 'flex'
    train_button_box.layout.display = 'none'

# Display the interface
display(controls)
display(output)

Output()